# ArcLoom Live — SPPU on PYNQ-Z2

Loads the ArcLoom bitstream, reads the Sharp distance sensor,
feeds it to the combinational loom fabric, reads decisions back.

The loom settles in ONE propagation delay. No clock cycles.
Write sensor → read decision. That's it.

In [ ]:
from pynq import Overlay, MMIO
from pynq.overlays.base import BaseOverlay
from pynq.lib.arduino import Arduino_Analog
import time

# Load ArcLoom overlay
# Requires arcloom.bit and arcloom.hwh in the same directory
ol = Overlay('/home/xilinx/arcloom.bit')
print('ArcLoom overlay loaded')
print(ol.ip_dict)

In [ ]:
# Get the ArcLoom AXI register base address
arcloom = ol.arcloom_0
# Or if using MMIO directly:
# base_addr = ol.ip_dict['arcloom_0']['phys_addr']
# arcloom_mmio = MMIO(base_addr, 16)  # 4 registers x 4 bytes

# Also load base overlay for Arduino analog sensor
# Note: can't load two overlays at once. Instead, use XADC directly.
# For now, we'll read the XADC through the PS.
from pynq import MMIO

# XADC base address on Zynq
XADC_BASE = 0xF8007100
xadc = MMIO(XADC_BASE, 0x100)

In [ ]:
# ---- Register Map ----
# 0x00 (W): sensor_adc[11:0] + valid[16]
# 0x00 (R): decision[5:0] + flags[13:8]
# 0x04 (R): loom_state[31:0]
# 0x08 (R): loom_state[35:32] + n_effective[12:8] + omega[20:13]

REG_SENSOR = 0x00
REG_DECISION = 0x00
REG_LOOM_LO = 0x04
REG_LOOM_HI_L6 = 0x08

TRIT_MAP = {0b00: ' 0', 0b01: '+1', 0b10: '-1', 0b11: '??'}

def write_sensor(adc_val):
    """Write 12-bit sensor value to ArcLoom with valid pulse."""
    arcloom.write(REG_SENSOR, (1 << 16) | (adc_val & 0xFFF))

def read_decision():
    """Read decision and status from ArcLoom."""
    d = arcloom.read(REG_DECISION)
    return {
        'steer':     TRIT_MAP[(d >> 0) & 0x3],
        'speed':     TRIT_MAP[(d >> 2) & 0x3],
        'conf':      TRIT_MAP[(d >> 4) & 0x3],
        'SL1':       bool(d & (1 << 8)),
        'safe_mode': bool(d & (1 << 9)),
        'dsf_valid': bool(d & (1 << 10)),
        'dsf_D':     TRIT_MAP[(d >> 11) & 0x3],
        'dsf_R_rev': bool(d & (1 << 13)),
    }

def read_loom_state():
    """Read full 18-trit loom state."""
    lo = arcloom.read(REG_LOOM_LO)
    hi_l6 = arcloom.read(REG_LOOM_HI_L6)
    state = lo | ((hi_l6 & 0xF) << 32)
    trits = []
    for i in range(18):
        trits.append(TRIT_MAP[(state >> (2*i)) & 0x3])
    return trits

def read_l6():
    """Read L6 topological constraint status."""
    hi_l6 = arcloom.read(REG_LOOM_HI_L6)
    n_eff = (hi_l6 >> 8) & 0x1F
    omega_val = (hi_l6 >> 13) & 0xFF
    return {'n_effective': n_eff, 'omega': omega_val, 'omega_pct': omega_val / 255.0 * 100}

print('ArcLoom register interface ready')

In [ ]:
# ---- Quick test: write a known value and read back ----
write_sensor(2048)  # mid-range
time.sleep(0.001)   # let UF pipeline process

dec = read_decision()
trits = read_loom_state()
l6 = read_l6()

print(f"Decision: steer={dec['steer']} speed={dec['speed']} conf={dec['conf']}")
print(f"SL-1={dec['SL1']}  SafeMode={dec['safe_mode']}")
print(f"Loom: {' '.join(trits)}")
print(f"L6: n_eff={l6['n_effective']}  omega={l6['omega_pct']:.1f}%")

In [ ]:
# ---- Live sensor loop ----
# Read Sharp distance sensor via XADC, feed to ArcLoom, display decisions

# XADC channel 0 (Arduino AR0) — read raw 12-bit value
# On Zynq, XADC auxiliary channel 0 is at offset 0x10 in the XADC status regs
# The data is in bits [15:4] of the register (12-bit left-justified)

def read_xadc_raw(channel=0):
    """Read raw 12-bit XADC value from auxiliary channel."""
    # Aux channel registers start at offset 0x240 in XADC address space
    # Each channel is 4 bytes apart
    val = xadc.read(0x40 + channel * 4)  # status register for aux channel
    return (val >> 4) & 0xFFF  # 12-bit right-justified

print('Starting live sensor loop... press Ctrl+C to stop')
try:
    while True:
        # Read sensor
        adc = read_xadc_raw(0)
        
        # Feed to ArcLoom
        write_sensor(adc)
        
        # Read decision
        dec = read_decision()
        l6 = read_l6()
        
        # Display
        bar = '=' * (adc // 64)  # visual bar
        sl1 = 'LOCK' if dec['SL1'] else '    '
        print(f"\rADC:{adc:4d} {bar:64s} "
              f"steer={dec['steer']} speed={dec['speed']} "
              f"n_eff={l6['n_effective']:2d} {sl1}", end='', flush=True)
        
        time.sleep(0.05)  # 20 Hz
except KeyboardInterrupt:
    print('\nStopped.')

In [ ]:
# ---- Motor control (TB6612FNG on Pmod A) ----
# Pins: AIN1=pmoda[0], AIN2=pmoda[1], BIN1=pmoda[2], BIN2=pmoda[3]
# PWM via PWMA/PWMB on pmoda[4], pmoda[5] (or tie high for full speed)
# STBY tied high (always on)

from pynq.lib.pmod import Pmod_IO

# Motor A (left)
ain1 = Pmod_IO(ol.PMODA, 0, 'out')
ain2 = Pmod_IO(ol.PMODA, 1, 'out')
# Motor B (right)
bin1 = Pmod_IO(ol.PMODA, 2, 'out')
bin2 = Pmod_IO(ol.PMODA, 3, 'out')

def motor_stop():
    ain1.write(0); ain2.write(0)
    bin1.write(0); bin2.write(0)

def motor_forward():
    ain1.write(1); ain2.write(0)
    bin1.write(1); bin2.write(0)

def motor_left():
    ain1.write(0); ain2.write(1)  # left motor reverse
    bin1.write(1); bin2.write(0)  # right motor forward

def motor_right():
    ain1.write(1); ain2.write(0)  # left motor forward
    bin1.write(0); bin2.write(1)  # right motor reverse

def motor_reverse():
    ain1.write(0); ain2.write(1)
    bin1.write(0); bin2.write(1)

print('Motor control ready (TB6612FNG on Pmod A)')
motor_stop()

In [ ]:
# ---- ArcLoom autonomous loop ----
# The loom decides. The motors obey.

TRIT_POS = '+1'
TRIT_NEG = '-1'
TRIT_NULL = ' 0'

print('ArcLoom autonomous mode... Ctrl+C to stop')
try:
    while True:
        # Sense
        adc = read_xadc_raw(0)
        write_sensor(adc)
        
        # Decide (loom settles in ~20ns, we're reading microseconds later)
        dec = read_decision()
        
        # Act
        if dec['steer'] == TRIT_POS:
            motor_right()
        elif dec['steer'] == TRIT_NEG:
            motor_left()
        elif dec['speed'] == TRIT_POS:
            motor_forward()
        elif dec['speed'] == TRIT_NEG:
            motor_reverse()
        else:
            motor_stop()
        
        # SafeMode override
        if dec['safe_mode']:
            motor_stop()
        
        time.sleep(0.02)  # 50 Hz control loop
except KeyboardInterrupt:
    motor_stop()
    print('\nStopped. Motors off.')